In [1]:
# import required libs
import numpy as np
import sys
sys.path.append(r"../Communiaction")
sys.path.append(r"../Utils")
import TCP_IP_UTILS
import Utils
import time

In [2]:
#Add power supply code here -> DC Sources
import sys
import pyvisa
sys.path.append(r"../PowerSupply")
import PS_Utils
Device_ID_DC = "USB0::0x2A8D::0x1202::MY59003914::0::INSTR"
rm = pyvisa.ResourceManager()
resources = rm.list_resources()
print("Available VISA resources:")
for r in resources:
        print(" -", r)
device_dc = PS_Utils.connect_E36313A(Device_ID_DC)
DIG_LS = 1
ANA_LS = 2
MAIN = 3

Available VISA resources:
 - USB0::0x0957::0x1F01::MY53270560::0::INSTR
 - USB0::0x0957::0x1F01::MY57280340::0::INSTR
 - USB0::0x2A8D::0x1202::MY59003914::0::INSTR
 - USB0::0x2A8D::0x9201::MY61391394::0::INSTR
 - USB0::0x2A8D::0x9201::MY61391395::0::INSTR
Connected to: Keysight Technologies,E36313A,MY59003914,2.1.0-1.0.4-1.12


In [3]:
#connect to ethernet socket 
HOST = '192.168.1.10'  # Remote device IP (server IP)
PORT = 7               # Echo port
client_socket = TCP_IP_UTILS.tcp_connect(HOST,PORT)
#Set IOs to default before turning on powersupply
ret_data = Utils.Set_ChipIO_to_Default(client_socket)
print(ret_data)
time.sleep(1) #Wait for IOs to settle

[Client] Connected to 192.168.1.10:7
Default Signals Set
[128]


In [4]:
#First Turn on Digital Leval Shifters
PS_Utils.set_voltage_E36313A(device_dc,DIG_LS,3,0.1)
print(PS_Utils.get_values_E36313A(device_dc,DIG_LS))
#Set IOs to default before turning on powersupply
ret_data = Utils.Set_ChipIO_to_Default(client_socket)
print(ret_data)

['+2.99815900E+00', '+5.50900000E-03']
Default Signals Set
[128]


In [5]:
#Turn on Main Chip PS
PS_Utils.set_voltage_E36313A(device_dc,MAIN,3,0.1)
print(PS_Utils.get_values_E36313A(device_dc,MAIN))

['+2.99818600E+00', '+1.29310000E-02']


In [6]:
#Turn on CLK 
Device_ID_CLK = "USB0::0x0957::0x1F01::MY57280340::0::INSTR"
device_clk = PS_Utils.connect_N5173B(Device_ID_CLK)
PS_Utils.set_voltage_N5173B(device_clk,0.45)
PS_Utils.set_frequency_N5173B(device_clk,1000000000)
PS_Utils.enable_output_N5173B(device_clk)

Connected to: Agilent Technologies, N5173B, MY57280340, B.01.65


In [7]:
#Call DL EN Scan Chain here with data
scan_id = Utils.DL_EN_SC_LM["id"]
scan_len = Utils.DL_EN_SC_LM["len"]
scan_data = np.ones(scan_len,dtype=int).tolist()
#scan_data[1] = 1
#scan_data[4] = 1
ret_data = Utils.ScanIN_Data(client_socket,scan_id,scan_len,scan_data,1)
print(ret_data)

[72, 0]
Fn identifier Sent
Received: 128
All Data Sent
Received: 128
FN: 3
Received Data Size: 1
All Data Received
[128]


In [8]:
#Call DL EN Scan Chain here with data
scan_id = Utils.DL_EN_SC_RM["id"]
scan_len = Utils.DL_EN_SC_RM["len"]
scan_data = np.ones(scan_len,dtype=int).tolist()
#scan_data[1] = 1
#scan_data[4] = 1
ret_data = Utils.ScanIN_Data(client_socket,scan_id,scan_len,scan_data,1)
print(ret_data)

[72, 0]
Fn identifier Sent
Received: 128
All Data Sent
Received: 128
FN: 3
Received Data Size: 1
All Data Received
[128]


In [9]:
#Control Logic TD Data
import numpy as np
import random
import re
def scan_gen(time):
    SC_data = f"""
    IN_EXT_LOC = 1'b0;
    TDC_EN_EXT_LOC = 1'b0;
    ENCHG_EXT_LOC = 1'b0;
    ENTIME_EXT_LOC = 1'b0;
    PCHG_EXT_LOC = 1'b0;
    VDAC_CTRL_EXT_LOC = 1'b0;
    DEL_RST_EXT_LOC = 1'b0;
    TDC_COMPUTE_EXT_LOC = 1'b0;
    TDC_RST_EXT_LOC = 1'b0;
    VTC_EN_EXT_LOC = 1'b0;

    IN_LB = 6'd0;
    IN_UB = 6'd0; 
    TDC_EN_LB = 6'd5;
    TDC_EN_UB = 6'd13;
    ENCHG_LB = 6'd2;
    ENCHG_UB = 6'd3;
    ENTIME_LB = 6'd0;
    ENTIME_UB = 6'd0;
    PCHG_LB = 6'd1;
    PCHG_UB = 6'd4;
    VDAC_CTRL_LB = 6'd2;
    VDAC_CTRL_UB = 6'd15; 
    DEL_RST_LB = 6'd0;
    DEL_RST_UB = 6'd0;
    TDC_COMPUTE_LB = 6'd11;
    TDC_COMPUTE_UB = 6'd20;
    TDC_RST_LB = 6'd1;
    TDC_RST_UB = 6'd13;
    BL_NUM_TGT = 2'd3;
    BL_DONE_TIME = 6'd25;
    VTC_EN_LB = 6'd5;
    VTC_EN_UB = 6'd14;
    SAMPLE_EDGE_TIME = 6'd{time};

    CHG_TIME = 1'd1;
    OSC_DEL = 1'd0;
    """
    SC_order = ["BL_DONE_TIME","SAMPLE_EDGE_TIME", "BL_NUM_TGT", "BLANK_4", "IN_LB", "TDC_EN_LB", "ENCHG_LB", "IN_EXT_LOC","TDC_EN_EXT_LOC","ENCHG_EXT_LOC",\
                "ENTIME_EXT_LOC","PCHG_EXT_LOC","VDAC_CTRL_EXT_LOC", 'ENTIME_LB', 'PCHG_LB', 'VDAC_CTRL_LB', 'DEL_RST_LB', 'TDC_COMPUTE_LB', 'TDC_RST_LB', \
                'VTC_EN_LB',"DEL_RST_EXT_LOC","TDC_COMPUTE_EXT_LOC","TDC_RST_EXT_LOC","VTC_EN_EXT_LOC","CHG_TIME","OSC_DEL", "VTC_EN_UB","TDC_RST_UB","TDC_COMPUTE_UB",\
                "DEL_RST_UB","VDAC_CTRL_UB","PCHG_UB","ENTIME_UB","ENCHG_UB","TDC_EN_UB","IN_UB","BLANK_12_PISO"]

    lines = SC_data.splitlines()
    sc_signal = {}
    for line in lines:
        if line.strip():  # Skip empty lines
            # Handle 'd' (decimal) format
            match = re.match(r"(\w+) = (\d+)'d([0-9]+);", line.strip())
            if match:
                signal_name = match.group(1)
                num_bits = int(match.group(2))  # Number of bits
                signal_value = match.group(3)  # Decimal value


                # If there's only 1 bit, do not add the bit-range
                if num_bits == 1:
                    name = signal_name
                else:
                    # Format the name with the bit-range (e.g., VTC_EN_UB<[5:0]>)
                    name = "%s" % (signal_name) # Using 0-based index for the range

                # Format the decimal value to binary and pad to the correct number of bits
                binary_value = bin(int(signal_value))[2:].zfill(num_bits)
                sc_signal[name] = binary_value


            # Handle 'b' (binary) format
            elif "b" in line:  # e.g., BL_NUM_TGT = 2'b01;
                match = re.match(r"(\w+) = (\d+)'b([01]+);", line.strip())
                if match:
                    signal_name = match.group(1)
                    num_bits = int(match.group(2))  # Number of bits
                    signal_value = match.group(3)  # Binary value

                    # If there's only 1 bit, do not add the bit-range
                    if num_bits == 1:
                        name = signal_name
                    else:
                        # Format the name with the bit-range (e.g., BL_NUM_TGT<[1:0]> for 2 bits)
                        name = "%s" % (signal_name)

                    # Format the binary value to match the number of bits
                    binary_value = signal_value.zfill(num_bits)
                    sc_signal[name] = binary_value


    # print(sc_signal)
    output_list = []
    for signal in SC_order:
        if "BLANK" in signal:
            num_blanks = re.findall(r'\d+', signal)
            output_list += [0]*int(num_blanks[0])
        else:
            temp = sc_signal[signal][::-1]
            output_list += list(map(int,temp))
    print(sc_signal["SAMPLE_EDGE_TIME"])
    output_list = output_list[::-1] + [0]*30
    return output_list

def set_CS(cs):
    ret_data = 0
    if cs == 0:
        signal_array = [Utils.CS_0,Utils.CS_1]
        value_array = [0,0]
        ret_data = Utils.Set_Chip_IO(client_socket,signal_array,value_array,1)
    if cs == 1:
        signal_array = [Utils.CS_0,Utils.CS_1]
        value_array = [1,0]
        ret_data = Utils.Set_Chip_IO(client_socket,signal_array,value_array,1)
    if cs == 2:
        signal_array = [Utils.CS_0,Utils.CS_1]
        value_array = [0,1]
        ret_data = Utils.Set_Chip_IO(client_socket,signal_array,value_array,1)
    if cs == 3:
        signal_array = [Utils.CS_0,Utils.CS_1]
        value_array = [1,1]
        ret_data = Utils.Set_Chip_IO(client_socket,signal_array,value_array,1)
    return ret_data

In [10]:
#Call Scan IN here with data
scan_id = Utils.CONTROL_LOGIC_SC["id"]
scan_len = Utils.CONTROL_LOGIC_SC["len"]
scan_data = scan_gen(5)
ret_data = Utils.ScanIN_Data(client_socket,scan_id,scan_len,scan_data,1)
#print(ret_data)

000101
[162, 0]
Fn identifier Sent
Received: 128
All Data Sent
Received: 128
FN: 3
Received Data Size: 1
All Data Received


In [11]:
#Initialise SRAM with default data.
write_data = Utils.get_Default_Write_Data().tolist()
ret_data = Utils.Write_SRAM(client_socket,write_data,1)
print(ret_data)

Fn identifier Sent
Received: 128
All Data Sent
Received: 128
FN: 6
Received Data Size: 1
All Data Received
[128]


In [12]:
#SRAM IMC DATA
arr = np.zeros((320, 1156),dtype=np.uint8)
#Write all reference lines at 128
for refline in range(24,280,36):
    #BANK 0
    arr[refline][767:767+128] = 1 #BL3
    arr[refline+1][767:767+128] = 1 #BL2
    arr[refline+2][767:767+128] = 1 #BL1
    arr[refline+3][767:767+128] = 1 #BL0

    #BANK 1
    arr[refline][511:511+128] = 1 #BL3
    arr[refline+1][511:511+128] = 1 #BL2
    arr[refline+2][511:511+128] = 1 #BL1
    arr[refline+3][511:511+128] = 1 #BL0

    #BANK 2
    arr[refline][255:255+128] = 1 #BL3
    arr[refline+1][255:255+128] = 1 #BL2
    arr[refline+2][255:255+128] = 1 #BL1
    arr[refline+3][255:255+128] = 1 #BL0

    #BANK 3
    arr[refline][0:0+128] = 1 #BL3
    arr[refline+1][0:0+128] = 1 #BL2
    arr[refline+2][0:0+128] = 1 #BL1
    arr[refline+3][0:0+128] = 1 #BL0
arr = arr.flatten(order='F')


In [13]:
#Initialise SRAM with default data.
write_data = arr.tolist()
ret_data = Utils.Write_SRAM_Masked(client_socket,write_data,1)
print(ret_data)

Fn identifier Sent
Received: 128
All Data Sent
Received: 128
FN: 10
Received Data Size: 1
All Data Received
[128]


In [14]:
#Turn on Analog PS
PS_Utils.set_voltage_E36313A(device_dc,ANA_LS,3,0.1)
print(PS_Utils.get_values_E36313A(device_dc,ANA_LS))

['+2.99815000E+00', '+2.52860000E-02']


In [15]:
signal_array = [Utils.BANK_EN_0,Utils.BANK_EN_1,Utils.BANK_EN_2,Utils.BANK_EN_3,Utils.CTRL_EN,Utils.InputEN_DAC]
value_array = [1,1,1,1,1,0]
ret_data = Utils.Set_Chip_IO(client_socket,signal_array,value_array,1)
print(ret_data)

Fn identifier Sent
Received: 128
All Data Sent
Received: 128
FN: 1
Received Data Size: 1
All Data Received
[128]


In [16]:
signal_array = [Utils.DFF_RST]
value_array = [1]
ret_data = Utils.Set_Chip_IO(client_socket,signal_array,value_array,1)
print(ret_data)

Fn identifier Sent
Received: 128
All Data Sent
Received: 128
FN: 1
Received Data Size: 1
All Data Received
[128]


In [17]:
TDC_DATA = []
signal_array = [Utils.SCN_SEL_2,Utils.SCN_SEL_1,Utils.SCN_SEL_0]
value_array = Utils.TDC_OUT_SC_LM["value"] #Scan chain id
ret_data = Utils.Set_Chip_IO(client_socket,signal_array,value_array,1)
print(ret_data)
for Bank_Sel in [0,1]:
    for cs in range(0,4):
        #select the BL
        ret_data = set_CS(cs)
        #Turn On IN_EN
        signal_array = [Utils.IN_EN,Utils.BANK_SEL,Utils.SCN_IN]
        value_array = [1,Bank_Sel,0]
        ret_data = Utils.Set_Chip_IO(client_socket,signal_array,value_array,1)
        #Sample the CLK_A 1
        signal_array = [Utils.CLK_A]
        value_array = [1]
        ret_data = Utils.Set_Chip_IO(client_socket,signal_array,value_array,1)
        #Sample the CLK_A 0
        signal_array = [Utils.CLK_A]
        value_array = [0]
        ret_data = Utils.Set_Chip_IO(client_socket,signal_array,value_array,1)
        #Sample the IN_EN 0
        signal_array = [Utils.IN_EN]
        value_array = [0]
        ret_data = Utils.Set_Chip_IO(client_socket,signal_array,value_array,1)
        #Call Scan Out
        scan_id = Utils.TDC_OUT_SC_LM["id"]
        scan_len = Utils.TDC_OUT_SC_LM["len"]
        ret_data = Utils.ScanOUT_Data(client_socket,scan_id,scan_len,1)
        TDC_DATA.append(ret_data)
        print(ret_data) 

Fn identifier Sent
Received: 128
All Data Sent
Received: 128
FN: 1
Received Data Size: 1
All Data Received
[128]
Fn identifier Sent
Received: 128
All Data Sent
Received: 128
FN: 1
Received Data Size: 1
All Data Received
Fn identifier Sent
Received: 128
All Data Sent
Received: 128
FN: 1
Received Data Size: 1
All Data Received
Fn identifier Sent
Received: 128
All Data Sent
Received: 128
FN: 1
Received Data Size: 1
All Data Received
Fn identifier Sent
Received: 128
All Data Sent
Received: 128
FN: 1
Received Data Size: 1
All Data Received
Fn identifier Sent
Received: 128
All Data Sent
Received: 128
FN: 1
Received Data Size: 1
All Data Received
[128, 2]
Fn identifier Sent
Received: 128
All Data Sent
Received: 128
FN: 4
Received Data Size: 80
All Data Received
[225, 134, 31, 110, 4, 17, 68, 16, 126, 120, 17, 68, 24, 110, 4, 225, 134, 31, 65, 4, 225, 132, 31, 118, 216, 225, 132, 31, 94, 248, 17, 132, 23, 65, 248, 17, 69, 16, 126, 248, 17, 70, 20, 65, 248, 17, 132, 21, 81, 132, 225, 71, 16, 81

In [18]:
signal_array = [Utils.SCN_SEL_2,Utils.SCN_SEL_1,Utils.SCN_SEL_0]
value_array = Utils.TDC_OUT_SC_RM["value"] #Scan chain id
ret_data = Utils.Set_Chip_IO(client_socket,signal_array,value_array,1)
print(ret_data)
for Bank_Sel in [0,1]:
    for cs in range(0,4):
        #select the BL
        ret_data = set_CS(cs)
        #Turn On IN_EN
        signal_array = [Utils.IN_EN,Utils.BANK_SEL,Utils.SCN_IN]
        value_array = [1,Bank_Sel,0]
        ret_data = Utils.Set_Chip_IO(client_socket,signal_array,value_array,1)
        #Sample the CLK_A 1
        signal_array = [Utils.CLK_A]
        value_array = [1]
        ret_data = Utils.Set_Chip_IO(client_socket,signal_array,value_array,1)
        #Sample the CLK_A 0
        signal_array = [Utils.CLK_A]
        value_array = [0]
        ret_data = Utils.Set_Chip_IO(client_socket,signal_array,value_array,1)
        #Sample the IN_EN 0
        signal_array = [Utils.IN_EN]
        value_array = [0]
        ret_data = Utils.Set_Chip_IO(client_socket,signal_array,value_array,1)
        #Call Scan Out
        scan_id = Utils.TDC_OUT_SC_RM["id"]
        scan_len = Utils.TDC_OUT_SC_RM["len"]
        ret_data = Utils.ScanOUT_Data(client_socket,scan_id,scan_len,1)
        TDC_DATA.append(ret_data)
        print(ret_data) 

Fn identifier Sent
Received: 128
All Data Sent
Received: 128
FN: 1
Received Data Size: 1
All Data Received
[128]
Fn identifier Sent
Received: 128
All Data Sent
Received: 128
FN: 1
Received Data Size: 1
All Data Received
Fn identifier Sent
Received: 128
All Data Sent
Received: 128
FN: 1
Received Data Size: 1
All Data Received
Fn identifier Sent
Received: 128
All Data Sent
Received: 128
FN: 1
Received Data Size: 1
All Data Received
Fn identifier Sent
Received: 128
All Data Sent
Received: 128
FN: 1
Received Data Size: 1
All Data Received
Fn identifier Sent
Received: 128
All Data Sent
Received: 128
FN: 1
Received Data Size: 1
All Data Received
[128, 2]
Fn identifier Sent
Received: 128
All Data Sent
Received: 128
FN: 4
Received Data Size: 80
All Data Received
[17, 68, 18, 97, 68, 17, 69, 18, 73, 68, 225, 71, 20, 126, 248, 225, 133, 31, 97, 196, 225, 135, 23, 113, 68, 17, 70, 24, 81, 4, 161, 135, 17, 126, 152, 17, 134, 23, 78, 248, 17, 70, 20, 73, 132, 17, 71, 22, 81, 132, 145, 70, 20, 121, 

In [19]:
#Set IO to default and exit    
ret_data = Utils.Set_ChipIO_to_Default(client_socket)
print(ret_data)

Default Signals Set
[128]


In [20]:
def TDC_OUT_u8_to_dec(TDC_OUT):
  arr = np.array(TDC_OUT, dtype=np.uint8)
  binary_strings = np.vectorize(np.binary_repr)(arr, width=8)
  binary_strings_rev = np.vectorize(lambda x: x[::-1])(binary_strings)
  all_bits = ''.join(binary_strings_rev)

  groups_of_10 = [all_bits[i:i+10] for i in range(0, len(all_bits), 10)]

  def signed_10bit_to_int(bin_str):
    sign_bit = bin_str[0]
    magnitude = int(bin_str[1:], 2)
    return magnitude if sign_bit == '0' else -magnitude

  signed_integers = [signed_10bit_to_int(bits) for bits in groups_of_10]

  return signed_integers

In [21]:
for i in range(16):
    print(TDC_OUT_u8_to_dec(TDC_DATA[i]))

[-29, -31, -29, -32, -32, -32, -31, -30, -32, -33, -29, -32, -29, -31, -32, -32, -28, -31, -27, -27, -28, -31, -30, -31, -32, -30, -32, -31, -34, -32, -31, -31, -33, -34, -32, -31, -32, -26, -34, -33, -31, -32, -34, -32, -32, -32, -32, -30, -35, -37, -36, -35, -34, -34, -38, -34, -40, -42, -40, -39, -38, -39, -40, -42]
[-32, -34, -32, -34, -34, -36, -33, -32, -34, -37, -31, -36, -31, -33, -34, -35, -31, -33, -31, -30, -31, -33, -32, -33, -36, -33, -36, -34, -38, -35, -35, -34, -36, -38, -35, -34, -34, -30, -38, -36, -34, -33, -38, -36, -35, -36, -36, -33, -39, -40, -40, -39, -37, -37, -41, -38, -43, -45, -44, -42, -42, -43, -44, -46]
[-32, -34, -32, -34, -34, -36, -33, -32, -34, -36, -31, -35, -31, -33, -34, -35, -31, -33, -31, -30, -31, -33, -32, -33, -35, -32, -35, -34, -37, -35, -33, -33, -36, -38, -35, -34, -34, -30, -38, -36, -33, -33, -38, -35, -35, -35, -35, -32, -39, -39, -39, -39, -37, -36, -41, -37, -43, -45, -43, -42, -42, -42, -44, -45]
[-32, -34, -32, -34, -34, -36, -33, -

In [ ]:
arr_write = write_data
u8_array = []
for i in range(0, len(arr_write), 8):
    byte_bits = arr_write[i:i+8]
    value = sum(bit << j for j, bit in enumerate(byte_bits))  # index 0 is LSB
    u8_array.append(value)

print(u8_array)

In [ ]:
#Call Read SRAM here
SRAM_DATA = Utils.Read_SRAM(client_socket,1)
print(len(SRAM_DATA))
print(SRAM_DATA)

In [ ]:
if SRAM_DATA == u8_array:
    print("Lists are equal")
else:
    print("Lists are not equal")

In [ ]:
#Set IO to default and exit    
ret_data = Utils.Set_ChipIO_to_Default(client_socket)
print(ret_data)


In [22]:
PS_Utils.kill_channel_E36313A(device_dc,ANA_LS)
time.sleep(0.3)
PS_Utils.kill_channel_E36313A(device_dc,MAIN)
time.sleep(0.3)
PS_Utils.kill_channel_E36313A(device_dc,DIG_LS)
PS_Utils.disconnect_E36313A(device_dc)
time.sleep(0.3)
PS_Utils.kill_N5173B(device_clk)
PS_Utils.disconnect_N5173B(device_clk)

0